In [0]:
%sql
CREATE TABLE IF NOT EXISTS silver.silver_clean (
    order_id INT,
    customer_name STRING,
    order_date DATE,
    amount DOUBLE
);

In [0]:
%sql
INSERT INTO silver.silver_clean
SELECT *
FROM bronze.orders;

In [0]:
orders_clean=spark.table("silver.silver_clean")

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import col,row_number
w=Window.partitionBy("order_id").orderBy(col("order_date").desc())
orders_with_rn=orders_clean.withColumn("rn",row_number().over(w))
total_cleaned=orders_clean.count()
total_dedup=orders_with_rn.filter(col("rn")==1).count()
duplicate_exists=total_cleaned!=total_dedup

In [0]:
dbutils.jobs.taskValues.set(key="has_duplicate",value=duplicate_exists)